# 🏫 OptiPlua — Simulateur d'Emploi du Temps
### Moteur Heuristique v2.1 | 7 contraintes dures (C6 Labo + C7 Capacité corrigés)

---
**Objectif :** Générer un emploi du temps réaliste en respectant :
- ✅ Compatibilité Enseignant ↔ Matière ↔ Type d'établissement
- ✅ Compatibilité Salle ↔ Classe (capacité + type)
- ✅ Indisponibilités des enseignants
- ✅ Limite d'heures hebdomadaires par enseignant
- ✅ Anti-collision : prof, salle, classe ne peuvent pas être en double sur le même créneau
- ✅ Semaine complète (Lundi → Vendredi, matin + après-midi)
- ✅ Exportation enrichie → `raw_schedules_test.csv`

---
## Cellule 1 — Importations & Chargement des Données

In [13]:
import pandas as pd
import random
import ast
import time
from collections import defaultdict

random.seed(42)  # Reproductibilité

# ── Chargement ──────────────────────────────────────────────────────────────
print("[1/4] Chargement des données...")
df_enseignants = pd.read_csv('enseignants_data.csv')
df_salles      = pd.read_csv('salles_data.csv')
df_matieres    = pd.read_csv('matieres_data.csv')
df_classes     = pd.read_csv('classes_data.csv')

# ── Parsing des colonnes liste (stockées comme string) ──────────────────────
def safe_parse_list(val):
    """Convertit une chaîne type "['A','B']" en liste Python."""
    try:
        return ast.literal_eval(val) if isinstance(val, str) else []
    except (ValueError, SyntaxError):
        return []

df_enseignants['Matieres']             = df_enseignants['Matieres'].apply(safe_parse_list)
df_enseignants['Niveaux_Autorises']    = df_enseignants['Niveaux_Autorises'].apply(safe_parse_list)
df_enseignants['Creneaux_Indisponibles'] = df_enseignants['Creneaux_Indisponibles'].apply(safe_parse_list)

print(f"   ✓ {len(df_classes)} classes | {len(df_enseignants)} enseignants | {len(df_salles)} salles | {len(df_matieres)} matières")
print("[1/4] OK")

[1/4] Chargement des données...
   ✓ 150 classes | 200 enseignants | 80 salles | 23 matières
[1/4] OK


---
## Cellule 2 — Définition des Créneaux & Index Rapides

In [14]:
print("[2/4] Construction des index...")

# ── Semaine complète ─────────────────────────────────────────────────────────
JOURS   = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi']
HEURES  = ['08:00-10:00', '10:00-12:00', '14:00-16:00', '16:00-18:00']
CRENEAUX = [f"{j}_{h}" for j in JOURS for h in HEURES]  # 20 créneaux

print(f"   ✓ {len(CRENEAUX)} créneaux disponibles (semaine complète)")

# ── Index : Type_Etablissement → Matières disponibles ───────────────────────
matieres_par_type = defaultdict(list)
for _, row in df_matieres.iterrows():
    matieres_par_type[row['Type_Etablissement']].append(row['ID_Matiere'])

# ── Index : ID_Matiere → infos matière ──────────────────────────────────────
matiere_info = df_matieres.set_index('ID_Matiere').to_dict('index')

# ── Index : Type_Etablissement → salles compatibles ─────────────────────────
salles_par_type = defaultdict(list)
for _, row in df_salles.iterrows():
    salles_par_type[row['Type_Etablissement']].append(row.to_dict())

# ── Index rapide enseignants : (type_etab, matiere) → liste de profs ────────
profs_par_matiere = defaultdict(list)
for _, row in df_enseignants.iterrows():
    for mat in row['Matieres']:
        profs_par_matiere[(row['Type_Etablissement'], mat)].append(row.to_dict())

print("   ✓ Index construits")
print("[2/4] OK")

[2/4] Construction des index...
   ✓ 20 créneaux disponibles (semaine complète)
   ✓ Index construits
[2/4] OK


---
## Cellule 3 — Contraintes Dures & Moteur Heuristique

In [15]:
print("[3/4] Initialisation du moteur de simulation...")

# ────────────────────────────────────────────────────────────────────────────
# CONTRAINTES DURES
# ────────────────────────────────────────────────────────────────────────────
def verifier_contraintes(id_prof, id_salle, id_classe, creneau,
                         heures_prof, indispo_prof,
                         max_heures_prof,
                         occupations_prof, occupations_salle, occupations_classe):
    """
    Retourne (True, '') si le placement est valide, sinon (False, raison).
    Contraintes vérifiées :
      C1 - Prof déjà occupé à ce créneau
      C2 - Salle déjà réservée à ce créneau
      C3 - Classe déjà assignée à ce créneau
      C4 - Créneau dans les indisponibilités du prof
      C5 - Limite d'heures hebdomadaires dépassée
    """
    if creneau in occupations_prof.get(id_prof, set()):
        return False, 'C1_prof_occupe'
    if creneau in occupations_salle.get(id_salle, set()):
        return False, 'C2_salle_occupee'
    if creneau in occupations_classe.get(id_classe, set()):
        return False, 'C3_classe_occupee'
    if creneau in indispo_prof:
        return False, 'C4_indispo'
    if heures_prof.get(id_prof, 0) >= max_heures_prof:
        return False, 'C5_max_heures'
    return True, ''


# ────────────────────────────────────────────────────────────────────────────
# MOTEUR PRINCIPAL
# ────────────────────────────────────────────────────────────────────────────
# C6 — Types de salles considérés comme 'laboratoire'
TYPES_LABO = {'Labo_Science', 'Labo_Informatique', 'Laboratoire'}

def generer_emploi_du_temps(df_classes, max_retries=200):
    """
    Génère un emploi du temps heuristique complet.
    Pour chaque classe, tente d'assigner UNE session par matière compatible.
    """
    emploi       = []          # Résultat final
    echecs       = []          # Classes/matières non assignées
    stats_raison = defaultdict(int)  # Compteur des causes d'échec

    # Structures de suivi des occupations (O(1) lookup)
    occupations_prof   = defaultdict(set)
    occupations_salle  = defaultdict(set)
    occupations_classe = defaultdict(set)
    heures_prof        = defaultdict(int)  # Heures assignées par prof cette semaine

    classes_shuffled = df_classes.sample(frac=1, random_state=42).to_dict('records')

    for classe in classes_shuffled:
        id_classe   = classe['ID_Classe']
        type_etab   = classe['Type_Etablissement']
        niveau      = classe['Niveau']
        nb_etudiants = classe['Nombre_Etudiants']

        matieres_dispo = matieres_par_type.get(type_etab, [])
        if not matieres_dispo:
            echecs.append({'ID_Classe': id_classe, 'Raison': 'aucune_matiere_type'})
            continue

        # On tente d'assigner UNE matière par classe
        id_matiere_cible = random.choice(matieres_dispo)
        nom_matiere      = matiere_info[id_matiere_cible]['Nom_Matiere']
        necessite_labo   = matiere_info[id_matiere_cible]['Necessite_Labo']

        # Profs compatibles : même type_etab + enseigne cette matière + niveau autorisé
        profs_compat = [
            p for p in profs_par_matiere.get((type_etab, nom_matiere), [])
            if niveau in p['Niveaux_Autorises']
        ]
        if not profs_compat:
            echecs.append({'ID_Classe': id_classe, 'Raison': 'aucun_prof_compatible',
                           'ID_Matiere': id_matiere_cible})
            continue

        # ── C6 : Contrainte Laboratoire ──────────────────────────────────────
        salles_du_type = salles_par_type.get(type_etab, [])
        if necessite_labo:
            salles_filtrees = [s for s in salles_du_type if s['Type_Salle'] in TYPES_LABO]
            if not salles_filtrees:
                # Aucun labo disponible pour ce type d'établissement → échec dur
                echecs.append({'ID_Classe': id_classe,
                               'Raison': 'C6_aucun_labo_disponible',
                               'ID_Matiere': id_matiere_cible})
                continue
        else:
            salles_filtrees = salles_du_type

        # ── C7 : Contrainte Capacité (DURE — fallback documenté) ─────────────
        salles_compat = [s for s in salles_filtrees if s['Capacite'] >= nb_etudiants]
        if not salles_compat:
            # Fallback DOCUMENTÉ : on prend les 3 plus grandes salles disponibles
            # et on comptabilise le cas dans stats_raison pour l'EDA.
            salles_compat = sorted(salles_filtrees,
                                   key=lambda s: s['Capacite'],
                                   reverse=True)[:3]
            stats_raison['C7_capacite_insuffisante_fallback'] += 1
        if not salles_compat:
            echecs.append({'ID_Classe': id_classe, 'Raison': 'C7_aucune_salle_compatible'})
            continue

        # ── Boucle de tentatives ────────────────────────────────────────────
        assigne  = False
        tentatives = 0

        while not assigne and tentatives < max_retries:
            tentatives += 1
            creneau  = random.choice(CRENEAUX)
            prof     = random.choice(profs_compat)
            salle    = random.choice(salles_compat)

            id_prof  = prof['ID_Enseignant']
            id_salle = salle['ID_Salle']
            max_h    = prof['Heures_Max_Par_Semaine']
            indispo  = set(prof['Creneaux_Indisponibles'])

            valide, raison = verifier_contraintes(
                id_prof, id_salle, id_classe, creneau,
                heures_prof, indispo, max_h,
                occupations_prof, occupations_salle, occupations_classe
            )

            if valide:
                # Mettre à jour les structures d'occupation
                occupations_prof[id_prof].add(creneau)
                occupations_salle[id_salle].add(creneau)
                occupations_classe[id_classe].add(creneau)
                heures_prof[id_prof] += 2  # Chaque créneau = 2h

                # Extraire Jour et Heures depuis le créneau
                parts        = creneau.split('_')
                jour         = parts[0]
                heure_debut, heure_fin = parts[1].split('-')

                emploi.append({
                    'ID_Classe'      : id_classe,
                    'Type_Etablissement': type_etab,
                    'Niveau'         : niveau,
                    'Nb_Etudiants'   : nb_etudiants,
                    'ID_Matiere'     : id_matiere_cible,
                    'Nom_Matiere'    : nom_matiere,
                    'ID_Enseignant'  : id_prof,
                    'Nom_Enseignant' : prof['Nom_Enseignant'],
                    'ID_Salle'       : id_salle,
                    'Capacite_Salle' : salle['Capacite'],
                    'Type_Salle'     : salle['Type_Salle'],
                    'Creneau'        : creneau,
                    'Jour'           : jour,
                    'Heure_Debut'    : heure_debut,
                    'Heure_Fin'      : heure_fin,
                    'Nb_Tentatives'  : tentatives
                })
                assigne = True
            else:
                stats_raison[raison] += 1

        if not assigne:
            echecs.append({
                'ID_Classe': id_classe,
                'Raison'   : f'max_retries_{max_retries}',
                'ID_Matiere': id_matiere_cible
            })

    return pd.DataFrame(emploi), pd.DataFrame(echecs), dict(stats_raison)


print("   ✓ Moteur heuristique v2.1 prêt (C6+C7 corrigés)")
print("[3/4] OK")

[3/4] Initialisation du moteur de simulation...
   ✓ Moteur heuristique v2.1 prêt (C6+C7 corrigés)
[3/4] OK


---
## Cellule 4 — Exécution, Validation & Export

In [16]:
print("[4/4] Lancement de la simulation...")
print("=" * 60)

t0 = time.time()
df_emploi, df_echecs, stats_raison = generer_emploi_du_temps(df_classes, max_retries=300)
elapsed = time.time() - t0

total       = len(df_classes)
assignes    = len(df_emploi)
taux_succes = assignes / total * 100 if total > 0 else 0

print(f"\n{'━'*60}")
print(f"  📊 RÉSULTATS DE LA SIMULATION")
print(f"{'━'*60}")
print(f"  ✅ Cours assignés   : {assignes:>5} / {total}  ({taux_succes:.1f}%)")
print(f"  ❌ Échecs           : {len(df_echecs):>5}")
print(f"  ⏱  Temps d'exécut.  : {elapsed:.3f}s")
print(f"{'━'*60}")

if stats_raison:
    print("\n  📋 Causes de rejet (tentatives infructueuses) :")
    for raison, count in sorted(stats_raison.items(), key=lambda x: -x[1]):
        print(f"     {raison:<30} : {count}")

if len(df_echecs) > 0:
    print("\n  🔴 Détail des échecs :")
    print(df_echecs.to_string(index=False))

# ── Validation : aucun doublon (prof + créneau) ──────────────────────────────
print("\n  🔍 Validation des contraintes dures...")
doublons_prof  = df_emploi.duplicated(subset=['ID_Enseignant', 'Creneau']).sum()
doublons_salle = df_emploi.duplicated(subset=['ID_Salle', 'Creneau']).sum()
doublons_class = df_emploi.duplicated(subset=['ID_Classe', 'Creneau']).sum()
print(f"     Conflits prof/créneau  : {doublons_prof}  {'✅' if doublons_prof == 0 else '❌'}")
print(f"     Conflits salle/créneau : {doublons_salle}  {'✅' if doublons_salle == 0 else '❌'}")
print(f"     Conflits classe/créneau: {doublons_class}  {'✅' if doublons_class == 0 else '❌'}")

# ── Aperçu ───────────────────────────────────────────────────────────────────
print("\n  👁  Aperçu (5 premières lignes) :")
try:
    from IPython.display import display
    display(df_emploi[['ID_Classe','Nom_Matiere','Nom_Enseignant','ID_Salle','Creneau','Nb_Tentatives']].head())
except ImportError:
    print(df_emploi[['ID_Classe','Nom_Matiere','Nom_Enseignant','ID_Salle','Creneau']].head().to_string())

# ── Statistiques distribution ────────────────────────────────────────────────
print("\n  📈 Distribution par type d'établissement :")
print(df_emploi.groupby('Type_Etablissement').size().rename('Cours_Assignes').to_string())

print("\n  📅 Distribution par jour :")
print(df_emploi.groupby('Jour').size().rename('Cours').reindex(JOURS).to_string())

print("\n  📚 Distribution par matière (Top 10) :")
print(df_emploi.groupby('Nom_Matiere').size().rename('Cours').sort_values(ascending=False).head(10).to_string())

# ── Export CSV enrichi ───────────────────────────────────────────────────────
output_file = 'raw_schedules_test.csv'
df_emploi.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n{'='*60}")
print(f"  💾 Fichier exporté : '{output_file}'")
print(f"     Colonnes : {list(df_emploi.columns)}")
print(f"     Lignes   : {len(df_emploi)}")
print(f"{'='*60}")
print("  ✅ Simulation terminée — Douaa peut démarrer l'EDA !")

[4/4] Lancement de la simulation...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📊 RÉSULTATS DE LA SIMULATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ Cours assignés   :   137 / 150  (91.3%)
  ❌ Échecs           :    13
  ⏱  Temps d'exécut.  : 0.007s
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  📋 Causes de rejet (tentatives infructueuses) :
     C1_prof_occupe                 : 8
     C2_salle_occupee               : 5
     C4_indispo                     : 1

  🔴 Détail des échecs :
ID_Classe                   Raison              ID_Matiere
    C0069 C6_aucun_labo_disponible  SUB_CENTRE_SOUTIEN_SVT
    C0012 C6_aucun_labo_disponible  SUB_CENTRE_SOUTIEN_SVT
    C0028 C6_aucun_labo_disponible  SUB_CENTRE_SOUTIEN_SVT
    C0017    aucun_prof_compatible SUB_CENTRE_SOUTIEN_MATH
    C0001    aucun_prof_compatible SUB_CENTRE_SOUTIEN_MATH
    C0025 C6_aucun_labo_disponible  SUB_CENTRE_SOUTIEN_SVT
    C0026 C6_aucun_labo_disponible  S

,ID_Classe,Nom_Matiere,Nom_Enseignant,ID_Salle,Creneau,Nb_Tentatives
0,C0074,ANGLAIS,Prof_13,R076,Lundi_16:00-18:00,1
1,C0019,CHIMIE,Prof_78,R019,Mardi_16:00-18:00,1
2,C0119,ANALYSE,Prof_45,R071,Vendredi_10:00-12:00,1
3,C0079,SVT,Prof_21,R013,Lundi_10:00-12:00,1
4,C0077,PHYSIQUE,Prof_93,R077,Mardi_16:00-18:00,1



  📈 Distribution par type d'établissement :
Type_Etablissement
Centre_Soutien    40
Ecole_Standard    42
Universite        55

  📅 Distribution par jour :
Jour
Lundi       24
Mardi       35
Mercredi    26
Jeudi       26
Vendredi    26

  📚 Distribution par matière (Top 10) :
Nom_Matiere
PHYSIQUE       21
ANALYSE        14
FRANCAIS       13
DROIT_CIVIL    12
ANGLAIS        10
MATH           10
CHIMIE          9
ECONOMIE        8
SVT             7
RESEAUX         6

  💾 Fichier exporté : 'raw_schedules_test.csv'
     Colonnes : ['ID_Classe', 'Type_Etablissement', 'Niveau', 'Nb_Etudiants', 'ID_Matiere', 'Nom_Matiere', 'ID_Enseignant', 'Nom_Enseignant', 'ID_Salle', 'Capacite_Salle', 'Type_Salle', 'Creneau', 'Jour', 'Heure_Debut', 'Heure_Fin', 'Nb_Tentatives']
     Lignes   : 137
  ✅ Simulation terminée — Douaa peut démarrer l'EDA !
